# Chapter 5: Listening live

This notebook runs every piece of code in the chapter, from the saved results, in order. Read the chapter alongside it: [Listening live](https://books.sugiv.fyi/voice_applications/listening-live.html).


In [ ]:
# Run this cell first. In Google Colab it fetches the companion repo; anywhere else, run the notebook
# from a clone of it. Either way it moves into the book's folder and installs what the chapter needs.
import os
import pathlib
import sys

if "google.colab" in sys.modules and not pathlib.Path("/content/voices-of-the-future").exists():
    !git clone -q --depth 1 https://github.com/sugix/voices-of-the-future /content/voices-of-the-future
candidates = [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, pathlib.Path("/content/voices-of-the-future")]
root = next(p for p in candidates if (p / "voice" / "sim").is_dir())
os.chdir(root / "voice")
%pip install -q -r ../requirements/results.txt

## How long to wait before writing a word

In [ ]:
import json
import re
import sys

import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, "sim")
from scoring import entity_hits, word_errors  # Chapter 4's scoring
from wav import load  # Chapter 2's WAV reader

refs = {f"halfwheel/{r['id']}": r for r in map(json.loads, open("data/calls/manifest.jsonl"))}
refs |= {f"librispeech/{r['id']}": r for r in map(json.loads, open("data/librispeech/manifest.jsonl"))}
manifest = [json.loads(l) for l in open("data/ch5/manifest.jsonl")]
turns = [r for r in manifest if r["set"] == "turns"]
clips = [r for r in manifest if r["set"] == "bargein"]
nemo = json.load(open("data/ch5/capture_nemo.json"))
kyutai = json.load(open("data/ch5/capture_kyutai.json"))["stt-1b-en_fr"]
kyutai_2b6 = json.load(open("data/ch4/capture_ch4.json"))["results"]["kyutai-stt-2.6b-en"]["files"]


def score(texts):
    """WER on Halfwheel's phone calls and on LibriSpeech over the phone, and Halfwheel entities heard."""
    out = []
    for s in ["halfwheel", "librispeech"]:
        errs = [word_errors(refs[k]["written"], t) for k, t in texts.items() if k.startswith(s)]
        out.append(sum(e for e, _ in errs) / sum(n for _, n in errs))
    hits = [ok for k, t in texts.items() if k.startswith("halfwheel") for _, ok in entity_hits(refs[k], t)]
    return out[0], out[1], sum(hits)


systems = [("Parakeet-TDT 0.6B, whole call at once", np.nan, nemo["tdt offline"]["text"])]
for name, v in nemo.items():
    if name.startswith("tdt chunk"):
        chunk, right = re.findall(r"[\d.]+", name)
        systems.append((f"Parakeet-TDT 0.6B, {chunk} s chunks + {right} s look-ahead", v["latency_s"], v["text"]))
systems.append(("Parakeet realtime EOU 120M", 0.16,
                {k: v["final"].replace("<EOU>", " ") for k, v in nemo["eou120m"]["files"].items()}))
systems.append(("Kyutai STT 1B", 0.5, {k: v["text"] for k, v in kyutai.items()}))
systems.append(("Kyutai STT 2.6B", 2.5, {f"{s}/{i}": v["text"] for key, v in kyutai_2b6.items()
                                          for s, c, i in [key.split("/")] if c == "phone"}))
rows = sorted(((n, lat, *score(t)) for n, lat, t in systems), key=lambda r: (np.nan_to_num(r[1], nan=99), r[0]))
print(f"{'':58} {'waits':>6} {'Halfwheel':>10} {'LibriSpeech':>12} {'entities':>9}")
for n, lat, hw, ls, ent in rows:
    print(f"{n:58} {'all' if np.isnan(lat) else f'{lat:.2f} s':>6} {hw:10.1%} {ls:12.1%} {ent:6}/48")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
for label, pick, style in [("Parakeet-TDT, chunk = look-ahead", lambda n: re.search(r"([\d.]+) s chunks \+ \1 s", n), "o-"),
                           ("Parakeet-TDT, 0.16 s chunks", lambda n: "0.16 s chunks" in n, "s--")]:
    pts = sorted((lat, ls) for n, lat, hw, ls, ent in rows if "TDT" in n and pick(n))
    ax.plot(*zip(*pts), style, label=label)
for n, lat, hw, ls, ent in rows:
    if "TDT" not in n:
        ax.plot(lat, ls, "D", ms=7, label=n)
offline = next(r for r in rows if np.isnan(r[1]))
ax.axhline(offline[3], color="0.5", lw=0.8, ls=":", label="Parakeet-TDT, whole call")
ax.set(xlabel="how long the model waits before committing to a word (s)", ylabel="WER, LibriSpeech by phone",
       ylim=(0, 0.35), title="Accuracy against waiting time")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.legend(fontsize=7, loc="upper right")
fig.tight_layout()

In [ ]:
r = turns[0]
key = f"turns/{r['id']}"
print("What the caller said, and when:")
for p in r["phrases"]:
    print(f"  {p['start']:5.2f} to {p['end']:5.2f} s   {p['text']}")
print("\nWhen each streaming model had written each word:")


def arrivals(growing):
    """[(time, text so far)] -> each final word with the time its first letters appeared."""
    first = []
    for t, text in growing:
        first += [t] * (len(text.split()) - len(first))
    return list(zip(first, growing[-1][1].split()))


kyutai_growing, text = [], ""
for t, piece in kyutai[key]["pieces"]:
    text += piece
    kyutai_growing.append((t, text))
for name, growing in [("Kyutai 1B", kyutai_growing), ("EOU 120M", [(t, x.replace("<EOU>", "")) for t, x in nemo["eou120m"]["files"][key]["events"]])]:
    print(f"  {name:10}" + "  ".join(f"{w} {t:.2f}" for t, w in arrivals(growing)))

## Has the caller finished?

In [ ]:
SILERO, FRAME = nemo["silero"]["probs"], nemo["silero"]["frame_s"]


def silence_endpoint(key, wait, start=0.7):
    """End the turn once Silero has heard no speech for `wait` seconds."""
    quiet = 0.0
    for i, p in enumerate(SILERO[key]):
        t = (i + 1) * FRAME
        quiet = 0.0 if p > 0.5 else quiet + FRAME
        if t > start and quiet >= wait - 1e-9:
            return t
    return None


def evaluate(endpoint):
    """How many turns were cut off mid-sentence, and how long the rest waited after the caller finished."""
    cut, waits = [], []
    for r in turns:
        t = endpoint(f"turns/{r['id']}")
        if t is not None and t < r["turn_end"]:
            cut.append(r["id"])
        elif t is not None:
            waits.append(t - r["turn_end"])
    return len(cut), np.median(waits) if waits else np.nan, cut


pauses = [p["end"] - p["start"] for r in turns for p in r["pauses"]]
print(f"{len(turns)} turns, {len(pauses)} mid-turn pauses from {min(pauses):.1f} to {max(pauses):.1f} s\n")
print(f"{'rule':38} {'cut off':>8} {'then waits':>11}")
for wait in [0.3, 0.5, 0.8, 1.0, 1.2, 1.5]:
    n, w, _ = evaluate(lambda k, wait=wait: silence_endpoint(k, wait))
    print(f"{f'Silero, {wait} s of silence':38} {n:5}/16 {w:10.2f} s")

In [ ]:
def kyutai_endpoint(key, threshold=0.5, head=2, start=0.7):
    """Kyutai's rule: end the turn when head 2 (no speech within 2 s) passes the threshold."""
    for t, *heads in kyutai[key]["prs"]:
        if t > start and heads[head] > threshold:
            return t
    return None


def text_at(key, t):
    return "".join(w for tt, w in kyutai[key]["pieces"] if tt <= t)


UNFINISHED = set("a an the to for of on in at my is are and or but with from by your it's i'm".split())


def last_word_endpoint(key, short=0.5, long=1.5, start=0.7):
    """Wait `short` seconds, unless the transcript so far ends in a word that cannot end a sentence."""
    quiet = 0.0
    for i, p in enumerate(SILERO[key]):
        t = (i + 1) * FRAME
        quiet = 0.0 if p > 0.5 else quiet + FRAME
        if t <= start or quiet < short - 1e-9:
            continue
        words = re.findall(r"[a-z']+", text_at(key, t).lower())
        if (words and words[-1] not in UNFINISHED) or quiet >= long - 1e-9:
            return t
    return None


eou = lambda key: next((t for t, txt in nemo["eou120m"]["files"][key]["events"] if "<EOU>" in txt), None)
print(f"{'rule':38} {'cut off':>8} {'then waits':>11}")
for name, fn in [("Kyutai head 2 > 0.5 (Kyutai's default)", kyutai_endpoint),
                 ("Kyutai head 2 > 0.9", lambda k: kyutai_endpoint(k, 0.9)),
                 ("last word, 0.5 s or 1.5 s", last_word_endpoint)]:
    n, w, _ = evaluate(fn)
    print(f"{name:38} {n:5}/16 {w:10.2f} s")
late = sum(eou(f"turns/{r['id']}") is None or eou(f"turns/{r['id']}") > r["seconds"] - 0.4 for r in turns)
print(f"{'Parakeet EOU token':38} {'0':>5}/16   only when the audio ended, in {late}/16 turns")
r = turns[0]
gap = r["pauses"][0]
print(f"\n{r['id']}: pause from {gap['start']:.2f} to {gap['end']:.2f} s. Transcript after 0.5 s of it: "
      f"'{text_at('turns/t01', gap['start'] + 0.5).strip()}'")

In [ ]:
r = turns[0]
key = f"turns/{r['id']}"
x, sr = load(f"data/ch5/turns/{r['id']}.wav")
fig, ax = plt.subplots(2, 1, figsize=(8, 4), sharex=True)
ax[0].plot(np.arange(len(x)) / sr, x, lw=0.3, color="0.4")
for p in r["phrases"]:
    ax[0].text(p["start"], 0.95, p["text"], fontsize=7, va="top")
ax[0].set(title=f"{r['id']}: a caller reading a ticket number off a receipt", ylim=(-1, 1.1), yticks=[])
t_sil = (np.arange(len(SILERO[key])) + 1) * FRAME
ax[1].plot(t_sil, SILERO[key], lw=1, label="Silero: speech now")
prs = np.array(kyutai[key]["prs"])
ax[1].plot(prs[:, 0], prs[:, 3], lw=1, label="Kyutai head 2: silence ahead")
for name, t, c, y in [("Kyutai > 0.5", kyutai_endpoint(key), "C3", 0.85), ("Silero 0.5 s", silence_endpoint(key, 0.5), "C4", 0.6),
                      ("Silero 1.2 s", silence_endpoint(key, 1.2), "C2", 0.35)]:
    for a in ax:
        a.axvline(t, color=c, ls="--", lw=1)
    ax[1].text(t + 0.04, y, f"ends turn: {name}", color=c, fontsize=7)
ax[1].set(xlabel="seconds", ylabel="probability", ylim=(0, 1.05))
ax[1].legend(fontsize=7, loc="lower right")
fig.tight_layout()

## When the caller talks over the receptionist

In [ ]:
def voice_stop(probs, r, sustain=0.0):
    """Stop talking at the first moment Silero has heard `sustain` seconds of voice during our own speech."""
    run = 0.0
    for i, p in enumerate(probs):
        t = (i + 1) * FRAME
        run = run + FRAME if p > 0.5 else 0.0
        if r["prompt_start"] < t <= r["prompt_end"] and run >= max(sustain, FRAME) - 1e-9:
            return t
    return None


def barge_report(name, stop):
    caught, early, false = [], 0, {"backchannel": 0, "none": 0}
    for r in clips:
        t = stop(r)
        if r["kind"] == "barge-in" and t is not None:
            if t >= r["onset"]:
                caught.append(t - r["onset"])
            else:
                early += 1
        elif r["kind"] != "barge-in" and t is not None:
            false[r["kind"]] += 1
    delay = f"{np.median(caught):.2f} s" if caught else "-"
    print(f"{name:30} {len(caught):3}/4 {early:9}/4 {delay:>8} {false['backchannel']:8}/4 {false['none']:7}/4")


header = f"{'stop when':30} {'caught':>6} {'too early':>11} {'delay':>8} {'mm-hm':>10} {'echo':>9}"
print(header)
for name, s in [("any voice", 0.0), ("300 ms of voice", 0.3), ("600 ms of voice", 0.6)]:
    barge_report(name, lambda r, s=s: voice_stop(SILERO[f"bargein/{r['id']}"], r, s))
print("\nWhat Kyutai 1B heard while the receptionist was talking, with nobody else on the line:")
for r in clips:
    if r["kind"] == "none":
        print(f"  said:  {r['prompt'][:70]}\n  heard: {kyutai['bargein/' + r['id']]['text'][:70]}")
        break

In [ ]:
def cancel_echo(mic, ref, taps=32, mu=0.1, eps=1e-3):
    """Normalised LMS: learn how our own voice comes back down the line, and subtract it."""
    w = np.zeros(taps)
    pad = np.concatenate([np.zeros(taps - 1), ref])
    out = np.empty_like(mic)
    for n in range(len(mic)):
        x = pad[n:n + taps][::-1]  # the last `taps` samples we played, newest first
        e = mic[n] - w @ x  # what we hear, minus our prediction of our own echo
        w += mu * e * x / (x @ x + eps)  # nudge the echo model towards what we heard
        out[n] = e
    return out


def reference_for(r, n):
    """What the receptionist was playing, lined up with the call recording."""
    p, sr = load(f"data/ch5/prompts/p{(int(r['id'][1:]) - 1) % 4 + 1}.wav")
    ref = np.zeros(n, dtype=np.float32)
    i = int(r["prompt_start"] * sr)
    ref[i:i + len(p)] = p[: n - i]
    return ref


# --- demo: the four clips where nobody interrupts, so everything we hear during the prompt is echo
for r in [c for c in clips if c["kind"] == "none"]:
    mic, sr = load(f"data/ch5/bargein/{r['id']}.wav")
    clean = cancel_echo(mic, reference_for(r, len(mic)))
    span = slice(int((r["prompt_start"] + 1.0) * sr), int(r["prompt_end"] * sr))  # after a second of learning
    hiss = slice(int((r["prompt_end"] + 0.5) * sr), None)  # the open line once we stop talking
    db = lambda x: 10 * np.log10(np.mean(x**2))
    print(f"{r['id']}: while we talk {db(mic[span]):6.1f} dB, after cancelling {db(clean[span]):6.1f} dB, "
          f"line hiss alone {db(mic[hiss]):6.1f} dB")

In [ ]:
aec = json.load(open("data/ch5/capture_aec.json"))["probs"]
print(header)
for name, s in [("any voice", 0.0), ("300 ms of voice", 0.3), ("600 ms of voice", 0.6)]:
    barge_report(name, lambda r, s=s: voice_stop(aec[f"bargein/{r['id']}"], r, s))


print("\nHow long Silero heard a voice, once the echo was gone:")
for r in clips:
    if r["kind"] != "none":
        p = np.array(aec[f"bargein/{r['id']}"]) > 0.5
        t = (np.arange(len(p)) + 1) * FRAME
        heard = p[(t > r["onset"] - 0.1) & (t < r["offset"] + 0.6)].sum() * FRAME
        print(f"  {r['kind']:12} {r['text']:42} {heard:.2f} s")